<a href="https://colab.research.google.com/github/rekhaannapurna/Paddy-Disease-Detection/blob/main/Colab/MobileNetV2_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"rekhaannapurna","key":"9a0621a137474cd26bb17ba9d7fdaeac"}'}

In [ ]:
import os

os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!kaggle datasets list -s paddy

ref                                                                title                                                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-----------------------------------------------------------------  --------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
aman2000jaiswal/agriculture-crop-images                            Agriculture crop images                               62575817  2021-03-10 16:56:50.653000          15171        301  0.9411765        
abbas829/paddy-dataset                                             Paddy Dataset                                            21837  2026-03-24 14:10:30.400000            112         19  1                
ritikbompilwar/plantstressidentification                           Plant Stress Identification (Paddy Leaves)            19760543  2022-09-06 15:09:00.530000            553         16  0.7

In [ ]:
!kaggle datasets download -d imbikramsaha/paddy-doctor

Dataset URL: https://www.kaggle.com/datasets/imbikramsaha/paddy-doctor
License(s): CC0-1.0
100% 1.02G/1.02G [00:26<00:00, 41.5MB/s]



In [ ]:
!unzip -q paddy-doctor.zip -d /content/paddy-doctor

In [ ]:
!find /content/paddy-doctor -maxdepth 2 -type d

/content/paddy-doctor
/content/paddy-doctor/paddy-disease-classification
/content/paddy-doctor/paddy-disease-classification/test_images
/content/paddy-doctor/paddy-disease-classification/train_images
/content/paddy-doctor/paddy-disease-classification/.ipynb_checkpoints


In [ ]:
!kaggle datasets download -d imbikramsaha/paddy-doctor

Dataset URL: https://www.kaggle.com/datasets/imbikramsaha/paddy-doctor
License(s): CC0-1.0
paddy-doctor.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
import glob
import os
import pandas as pd

try:
    import fastkaggle
except ModuleNotFoundError:
    !pip install -Uq fastkaggle

from fastkaggle import *
from fastai.vision.all import *

set_seed(42)

# Our already-downloaded Paddy Doctor dataset
path = Path('/content/paddy-doctor/paddy-disease-classification')

# Train images
train_path = path / 'train_images'
train_files = get_image_files(train_path)

# Test images
test_path = path / 'test_images'
test_files = get_image_files(test_path).sorted()

# Check available files
print("Dataset path:", path)
print("Training images:", len(train_files))
print("Test images:", len(test_files))

# Train labels
train_df = pd.read_csv(path / 'train.csv')
print("Train CSV shape:", train_df.shape)

print("\nClass distribution:")
print(train_df.label.value_counts())

Dataset path: /content/paddy-doctor/paddy-disease-classification
Training images: 10407
Test images: 3469
Train CSV shape: (10407, 4)

Class distribution:
label
normal                      1764
blast                       1738
hispa                       1594
dead_heart                  1442
tungro                      1088
brown_spot                   965
downy_mildew                 620
bacterial_leaf_blight        479
bacterial_leaf_streak        380
bacterial_panicle_blight     337
Name: count, dtype: int64


In [ ]:
dblock = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    get_y=parent_label,
    splitter=RandomSplitter(0.2, seed=42),
    item_tfms=Resize(480, method='squish'),
    batch_tfms=aug_transforms(size=224, min_scale=0.75)
)

dls = dblock.dataloaders(train_path)

In [ ]:
dls = ImageDataLoaders.from_folder(
    train_path,
    valid_pct=0.2,
    seed=42,
    item_tfms=Resize(480, method='squish'),
    batch_tfms=aug_transforms(size=224, min_scale=0.75)
)

In [ ]:
from fastai.vision.all import *
import torchvision.models as models
import torch.nn as nn

model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

# Remove MobileNetV2's original classifier
model.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(model.last_channel, dls.c)
)

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 136MB/s]


In [ ]:
learn = Learner(
    dls,
    model,
    loss_func=CrossEntropyLossFlat(),
    metrics=error_rate
).to_fp16()

In [ ]:
learn.fine_tune(50)

epoch,train_loss,valid_loss,error_rate,time
0,0.985328,0.897398,0.272465,01:40


epoch,train_loss,valid_loss,error_rate,time
0,0.361245,0.288424,0.093224,01:24
1,0.241939,0.223203,0.069678,01:24
2,0.178391,0.186385,0.060067,01:24
3,0.144530,0.195291,0.059587,01:24
4,0.138427,0.195938,0.050937,01:25
5,0.123700,0.174541,0.049015,01:24
6,0.126823,0.260476,0.072081,01:24
7,0.145403,0.259615,0.069197,01:24
8,0.147231,0.215241,0.059587,01:26
9,0.142807,0.198919,0.056223,01:25


In [ ]:
import os

print(os.listdir('/content/drive'))

['MyDrive']


In [ ]:
from pathlib import Path

backup_path = Path('/content/drive/MyDrive/Paddy_Disease_Project')
backup_path.mkdir(parents=True, exist_ok=True)

print("Backup folder:", backup_path)

Backup folder: /content/drive/MyDrive/Paddy_Disease_Project


In [ ]:
learn.export(
    str(backup_path / 'resnet34_paddy_baseline.pkl')
)

print("Model exported successfully!")

Model exported successfully!


In [ ]:
learn.save(
    str(backup_path / 'resnet34_paddy_baseline')
)

print("Model weights saved successfully!")

Model weights saved successfully!


In [ ]:
!ls -lh /content/drive/MyDrive/Paddy_Disease_Project

total 36M
-rw-r--r-- 1 root root 9.4M Sep  4 15:51 resnet34_paddy_baseline.pkl
-rw-r--r-- 1 root root  26M Sep  4 15:54 resnet34_paddy_baseline.pth


In [ ]:
results = """
PADDY DISEASE DETECTION - PHASE 1 BASELINE

Dataset:
- Training images: 10,407
- Test images: 3,469
- Classes: 10

Model:
- ResNet34
- FastAI
- Resize: 480 -> 224
- Augmentation: aug_transforms(size=224, min_scale=0.75)
- Fine-tuning: 50 epochs
- Learning rate: 0.005
- Seed: 42

Validation:
- Accuracy: 98.03%
- Macro F1: 97.82%
- Weighted F1: 98.04%

TTA:
- Accuracy: 98.17%
- Macro F1: 97.98%
- Weighted F1: 98.18%

Phase 1: COMPLETE
Phase 2: NOT STARTED
"""

with open(backup_path / 'phase1_results.txt', 'w') as f:
    f.write(results)

print("Results saved!")

Results saved!


In [ ]:
!ls -lh /content/drive/MyDrive/Paddy_Disease_Project

total 36M
-rw-r--r-- 1 root root  459 Sep  4 15:54 phase1_results.txt
-rw-r--r-- 1 root root 9.4M Sep  4 15:51 resnet34_paddy_baseline.pkl
-rw-r--r-- 1 root root  26M Sep  4 15:54 resnet34_paddy_baseline.pth
